In [1]:
from ollama import chat
from ollama import ChatResponse

response: ChatResponse = chat(
    model='gemma4', 
    messages=[
        {
            'role': 'user',
            'content': 'You are a 9 year old student. write a letter to your cousin explaining how much you are enjoying your new class and new subjects this year.',
        },
    ],
    # Add options to enable logprobs
    logprobs=True,
    top_logprobs=3
)

print(response['message']['content'])
# or access fields directly from the response object
print(response.message.content)

# Access and loop through the logprobs data
logprobs = response.get('logprobs') or response.__dict__.get('logprobs')
if logprobs:
    print("\nToken Probabilities:")
    for item in logprobs:
        print(f"Token: {item.get('token')} | Logprob: {item.get('logprob')}")


Dear [Cousin's Name],

Guess what?! I wanted to write you a letter because school this year is SO much fun! Seriously, it’s way better than last year, and I wish you were here to see it.

I know we talked about me being scared of starting Grade 4, but I was totally wrong. Everything is really cool, and the subjects are actually super interesting.

My favorite class right now has to be Science! Our teacher, Mr. Jones, showed us this huge volcano model, and he explained how lava works and how tectonic plates move. It felt like watching a documentary in real life—it was awesome! We even did an experiment where we mixed baking soda and vinegar to make it fizz really big, and it looked exactly right!

And guess what? Math isn't so boring anymore either! At first, I thought fractions would be tricky, but my teacher helped us learn about pizza slices and dividing them. It makes way more sense when you can picture it, like we were actually cutting up a giant pepperoni pizza in the classroom! 🍕

In [6]:
def calculate_response_confidence(logprobs):
    """Calculate an overall confidence score for the response.
    Returns a score between 0 and 1, where:
    - 1 indicates very high confidence
    - 0 indicates very low confidence
    """
    if not logprobs:
        return None

    import math

    # Normalize to an iterable of items that contain logprob values
    if hasattr(logprobs, 'content'):
        items = logprobs.content
    elif isinstance(logprobs, dict) and 'content' in logprobs:
        items = logprobs['content']
    else:
        items = logprobs

    # Extract numeric logprob values from items
    lp_values = []
    for item in items:
        if isinstance(item, dict):
            lp = item.get('logprob') or item.get('log_prob') or item.get('logProbability')
        else:
            lp = getattr(item, 'logprob', None) or getattr(item, 'log_prob', None)
        if lp is None:
            try:
                lp = float(item)
            except Exception:
                continue
        try:
            lp_values.append(float(lp))
        except Exception:
            continue

    if not lp_values:
        return None

    # Convert log-probabilities to probabilities
    probs = [math.exp(lp) for lp in lp_values]

    # Convert logprobs to probabilities
    avg_confidence = sum(probs) / len(probs)
    min_confidence = min(probs)

    # Calculate metrics

    # Weight both average and minimum confidence in the final score
    # This helps catch both overall low confidence and individual uncertain tokens
    confidence_score = (0.7 * avg_confidence) + (0.3 * min_confidence)

    return round(confidence_score, 4)

In [7]:
confidence_score = calculate_response_confidence(logprobs)
print(f"\nOverall Response Confidence Score: {confidence_score}")


Overall Response Confidence Score: 0.6353
